In [ ]:
def generate_questions_with_rag(
        self,
        source: str,
        source_type: str,
        num: int,
        question_type: QuestionType = "Multiple Choice",
        prompt_template: Optional[str] = None,
        custom_instructions: Optional[str] = None,
        response_model: Optional[Type[Any]] = None,
        learning_objective: str = "",
        difficulty_level: str = "",
        output_format: Optional[OutputFormatType] = None,
        **kwargs
    ) -> Any:
        """Generate questions using RAG (Retrieval Augmented Generation)."""
        try:
            # Initialize embeddings if not already done
            if self.embeddings is None:
                self.embeddings = OpenAIEmbeddings()

            # Load and prepare data
            content = self._load_data(source, source_type)
            if not content:
                raise ValueError("No content loaded from source")

            # Setup RAG components
            vector_store = self._create_vector_store(content)
            qa_chain = self._setup_retrieval_qa(vector_store)

            # Get parser and model
            parser, model = self._get_parser_and_model(question_type, response_model)
            format_instructions = parser.get_format_instructions()

            # Get and validate prompt template
            template = self._get_prompt_template(question_type, prompt_template)
            if template is None:
                template = ""

            # Build the complete prompt
            full_template = f"""{template}
            Learning Objective: {{learning_objective}}
            Difficulty Level: {{difficulty_level}}

            Generate {num} questions based on the provided content.
            The response should be in JSON format.
            {{format_instructions}}
            """

            if custom_instructions:
                full_template += f"\n\nAdditional Instructions:\n{custom_instructions}"

            # Create prompt template
            question_prompt = PromptTemplate(
                input_variables=["num", "topic", "learning_objective", "difficulty_level"],
                template=full_template,
                partial_variables={"format_instructions": format_instructions}
            )

            # Format the query
            query = question_prompt.format(
                num=num,
                topic=content[:1000],  # Limit content length
                learning_objective=learning_objective,
                difficulty_level=difficulty_level,
                **kwargs
            )

            # Get results from QA chain
            results = qa_chain.invoke({"query": query, "n_results": 3})

            try:
                structured_output = parser.parse(results["result"])
                
                if output_format:
                    self._handle_output_format(structured_output, output_format)

                return structured_output

            except Exception as e:
                logger.error(f"Error parsing output: {str(e)}")
                logger.debug(f"Raw output: {results}")
                return model()

        except Exception as e:
            logger.error(f"Error in generate_questions_with_rag: {str(e)}")
            raise


In [1]:
! pip install git+https://github.com/Shubhwithai/educhain.git@ragtest

  Cloning https://github.com/Shubhwithai/educhain.git (to revision ragtest) to /private/var/folders/j2/5k7518y170q1wlfrqlw1vb240000gn/T/pip-req-build-stjyihl9
  Running command git clone --filter=blob:none --quiet https://github.com/Shubhwithai/educhain.git /private/var/folders/j2/5k7518y170q1wlfrqlw1vb240000gn/T/pip-req-build-stjyihl9
  on a case-insensitive filesystem) and only one from the same
  colliding group is in the working tree:

    'cookbook/starters/Educhain_Starter_Guide_V3.ipynb'
    'cookbook/starters/educhain_Starter_guide_v3.ipynb'
  Running command git checkout -b ragtest --track origin/ragtest
  Switched to a new branch 'ragtest'
  branch 'ragtest' set up to track 'origin/ragtest'.
  Resolved https://github.com/Shubhwithai/educhain.git to commit 8df0d965b74c164b669014a7940ab5f31fc05d16
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")



In [3]:
from educhain import Educhain

client = Educhain()

url_list = client.qna_engine.generate_questions_with_rag(
    source="https://www.buildfastwithai.com/genai-course",
    source_type="url",
    num=20,
    question_type="Multiple Choice",
    difficulty_level="Intermediate",
    custom_instructions= "Ask questions only about Satvik"

)

url_list.show()

Question 1:
Question: What degrees does Satvik hold?
Options:
  A. Bachelor's and Master's degrees from IIT Delhi
  B. PhD from Harvard
  C. Bachelor's degree from Stanford
  D. Master's degree from MIT

Correct Answer: Bachelor's and Master's degrees from IIT Delhi

Question 2:
Question: What is Satvik's role in Build Fast with AI?
Options:
  A. Founder
  B. Lead Instructor
  C. Marketing Manager
  D. Project Coordinator

Correct Answer: Founder

Question 3:
Question: How many students has Satvik taught?
Options:
  A. Over 15,000 students
  B. Around 5,000 students
  C. 10,000 students
  D. 20,000 students

Correct Answer: Over 15,000 students

Question 4:
Question: Which companies has Satvik collaborated with?
Options:
  A. Google, Microsoft, and BCG
  B. Apple and IBM
  C. Tesla and SpaceX
  D. Facebook and Amazon

Correct Answer: Google, Microsoft, and BCG

Question 5:
Question: What teaching approach does Satvik believe in?
Options:
  A. A theoretical approach
  B. A practical app

In [4]:
from educhain import Educhain

client = Educhain()

text_questions = client.qna_engine.generate_questions_with_rag(
    source="""Navigate the AI Landscape
            After Week 1, you'll possess a deep understanding of LLMs, Transformers, and Prompt Engineering, enabling you to guide AI initiatives with confidence.""",
    source_type="text",
    num=10,
    question_type="Multiple Choice",
    difficulty_level="Intermediate",
    custom_instructions= "Focus on LLMS"
)

text_questions.show()

Question 1:
Question: What does LLM stand for in the context of AI?
Options:
  A. Large Learning Model
  B. Large Language Model
  C. Limited Language Model
  D. Language Learning Model

Correct Answer: Large Language Model
Explanation: LLMs are a type of AI model designed to understand and generate human-like text.

Question 2:
Question: Which architecture is primarily used for developing LLMs?
Options:
  A. Convolutional Neural Networks
  B. Recurrent Neural Networks
  C. Transformers
  D. Decision Trees

Correct Answer: Transformers
Explanation: Transformers are the backbone of modern LLMs due to their ability to handle sequential data effectively.

Question 3:
Question: What is Prompt Engineering?
Options:
  A. Designing AI hardware
  B. Creating user interfaces
  C. Crafting effective prompts
  D. Building datasets

Correct Answer: The art of crafting effective prompts to get the best results from AI models.
Explanation: Prompt Engineering is crucial in maximizing the performance 

In [5]:
from educhain import Educhain

client = Educhain()


pdf_questions = client.qna_engine.generate_questions_with_rag(
    source="/Users/shubham/projects/educhain/NIPS-2017-attention-is-all-you-need-Paper.pdf",
    source_type="pdf",
    num=20,
    question_type="Multiple Choice",
    learning_objective="",
    difficulty_level="Intermediate",
    custom_instructions= "what is this pdf about"
)

pdf_questions.show()

Question 1:
Question: What is the primary architecture proposed in the paper 'Attention Is All You Need'?
Options:
  A. Recurrent Neural Network
  B. Convolutional Neural Network
  C. The Transformer
  D. Support Vector Machine

Correct Answer: The Transformer

Question 2:
Question: What does the Transformer architecture primarily rely on?
Options:
  A. Recurrent layers
  B. Convolutional layers
  C. Attention mechanisms
  D. Random Forests

Correct Answer: Attention mechanisms

Question 3:
Question: Which two machine translation tasks were used to test the Transformer model?
Options:
  A. English-to-Spanish and English-to-Italian
  B. English-to-German and English-to-French
  C. German-to-English and French-to-English
  D. Spanish-to-English and Italian-to-English

Correct Answer: English-to-German and English-to-French

Question 4:
Question: What advantage does the Transformer model have over traditional models?
Options:
  A. It is easier to implement.
  B. It is more parallelizable 